# 06 – Synthetic Loyalty, Rewards, Inventory, Campaign & Ledger Data




## 1. Import libraries

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Locate project and load RFM and segment files

In [2]:
current_path = Path.cwd()
project_root = None

for candidate in [current_path, *current_path.parents]:
    if (candidate / "data" / "processed").exists():
        project_root = candidate
        break

if project_root is None:
    raise FileNotFoundError(
        "Could not find data/processed. Open this notebook from inside your project folder."
    )

processed_dir = project_root / "data" / "processed"
rfm_file = processed_dir / "rfm_customer_features.csv"
segments_file = processed_dir / "customer_rfm_segments.csv"

for file_path in [rfm_file, segments_file]:
    if not file_path.exists():
        raise FileNotFoundError(f"Required input file not found: {file_path}")

rfm = pd.read_csv(rfm_file)
segments = pd.read_csv(segments_file)

print("RFM rows:", len(rfm))
print("Segment rows:", len(segments))
print("RFM customers:", rfm["customer_id"].nunique())
print("Segment customers:", segments["customer_id"].nunique())

display(rfm.head())
display(segments.head())

RFM rows: 5878
Segment rows: 5878
RFM customers: 5878
Segment customers: 5878


,customer_id,recency,frequency,monetary,last_purchase_date
0,12346.0,326,12,77556.46,2011-01-18 10:01:00
1,12347.0,3,8,4921.53,2011-12-07 15:52:00
2,12348.0,76,5,2019.40,2011-09-25 13:13:00
3,12349.0,19,4,4428.69,2011-11-21 09:51:00
4,12350.0,311,1,334.40,2011-02-02 16:01:00


,customer_id,recency,frequency,monetary,cluster
0,12346.0,326,12,77556.46,1
1,12347.0,3,8,4921.53,1
2,12348.0,76,5,2019.40,1
3,12349.0,19,4,4428.69,1
4,12350.0,311,1,334.40,0


## 3. Validate inputs and prepare output folder

In [3]:
required_rfm = {"customer_id", "recency", "frequency", "monetary"}
required_segments = {"customer_id", "cluster"}

if not required_rfm.issubset(rfm.columns):
    raise ValueError(f"RFM file must contain: {sorted(required_rfm)}")

if not required_segments.issubset(segments.columns):
    raise ValueError(f"Segment file must contain: {sorted(required_segments)}")

if not rfm["customer_id"].is_unique:
    raise ValueError("RFM file must have one row per customer.")

if not segments["customer_id"].is_unique:
    raise ValueError("Segment file must have one row per customer.")

if set(rfm["customer_id"]) != set(segments["customer_id"]):
    raise ValueError("RFM and segment files must contain the same customer IDs.")

output_dir = processed_dir / "synthetic"
output_dir.mkdir(parents=True, exist_ok=True)

# Merge only for assigning demo loyalty tiers; original input files are not modified.
customer_base = rfm.merge(
    segments[["customer_id", "cluster"]],
    on="customer_id",
    how="inner",
    validate="one_to_one"
)

print("Input validation: PASSED")
print("Output folder:", output_dir)

Input validation: PASSED
Output folder: d:\loyality_reward_optimization\data\processed\synthetic


## 4. Generate customer loyalty profiles

In [4]:
customer_profiles = customer_base[["customer_id", "cluster", "recency", "frequency", "monetary"]].copy()

# Cluster 1 was the higher-frequency/higher-monetary group in the baseline result.
# Use a reproducible random assignment within each group for demo tier variation.
tier_values = []
for cluster_value in customer_profiles["cluster"]:
    if cluster_value == 1:
        tier_values.append(rng.choice(["Gold", "Platinum"], p=[0.70, 0.30]))
    else:
        tier_values.append(rng.choice(["Bronze", "Silver"], p=[0.65, 0.35]))

customer_profiles["loyalty_tier"] = tier_values
customer_profiles["points_balance"] = rng.integers(100, 12001, size=len(customer_profiles))
customer_profiles["profile_status"] = rng.choice(
    ["active", "inactive"],
    size=len(customer_profiles),
    p=[0.95, 0.05]
)

profile_reference_date = pd.to_datetime(rfm["last_purchase_date"], errors="coerce").max()
if pd.isna(profile_reference_date):
    profile_reference_date = pd.Timestamp("2011-12-10")

customer_profiles["profile_created_at"] = (
    profile_reference_date.normalize()
    - pd.to_timedelta(rng.integers(30, 900, size=len(customer_profiles)), unit="D")
)

customer_profiles["is_synthetic"] = True

customer_loyalty_profiles = customer_profiles[
    [
        "customer_id",
        "loyalty_tier",
        "points_balance",
        "profile_status",
        "profile_created_at",
        "is_synthetic"
    ]
].copy()

display(customer_loyalty_profiles.head())
display(customer_loyalty_profiles["loyalty_tier"].value_counts().to_frame("customer_count"))

,customer_id,loyalty_tier,points_balance,profile_status,profile_created_at,is_synthetic
0,12346.0,Platinum,348,active,2009-12-20,True
1,12347.0,Gold,8579,active,2011-04-14,True
2,12348.0,Platinum,11795,active,2011-06-05,True
3,12349.0,Gold,3444,active,2010-04-11,True
4,12350.0,Bronze,2536,active,2009-11-15,True


,customer_count
loyalty_tier,
Bronze,2330
Gold,1659
Silver,1216
Platinum,673


## 5. Generate campaign budgets

In [5]:
campaign_ids = [f"SYN-CAMP-{i:03d}" for i in range(1, 6)]

campaign_budgets = pd.DataFrame({
    "campaign_id": campaign_ids,
    "budget_limit": [50000, 75000, 100000, 40000, 60000],
    "committed_amount": [5000, 12000, 15000, 3000, 8000],
    "approved_amount": [0, 0, 0, 0, 0],
    "campaign_status": ["active", "active", "active", "active", "scheduled"],
    "is_synthetic": True
})

campaign_budgets["available_budget_before_approval"] = (
    campaign_budgets["budget_limit"] - campaign_budgets["committed_amount"]
)

display(campaign_budgets)

,campaign_id,budget_limit,committed_amount,approved_amount,campaign_status,is_synthetic,available_budget_before_approval
0,SYN-CAMP-001,50000,5000,0,active,True,45000
1,SYN-CAMP-002,75000,12000,0,active,True,63000
2,SYN-CAMP-003,100000,15000,0,active,True,85000
3,SYN-CAMP-004,40000,3000,0,active,True,37000
4,SYN-CAMP-005,60000,8000,0,scheduled,True,52000


## 6. Generate inventory snapshot

In [6]:
product_ids = [f"SYN-PROD-{i:03d}" for i in range(1, 31)]

on_hand = rng.integers(20, 501, size=len(product_ids))
reserved = np.array([rng.integers(0, int(stock) + 1) for stock in on_hand])
reorder_level = rng.integers(10, 81, size=len(product_ids))

inventory_snapshot = pd.DataFrame({
    "product_id": product_ids,
    "on_hand_stock": on_hand,
    "reserved_stock": reserved,
    "reorder_level": reorder_level,
    "replenishment_date": pd.NaT,
    "inventory_updated_at": pd.Timestamp("2011-12-10 00:00:00"),
    "is_synthetic": True
})

inventory_snapshot["available_stock"] = (
    inventory_snapshot["on_hand_stock"] - inventory_snapshot["reserved_stock"]
)

# Add a few future replenishment dates to demonstrate replenishment constraints.
for idx in [2, 7, 12, 18, 24]:
    inventory_snapshot.loc[idx, "replenishment_date"] = (
        pd.Timestamp("2011-12-10") + pd.Timedelta(days=int(rng.integers(2, 15)))
    )

display(inventory_snapshot.head(10))

,product_id,on_hand_stock,reserved_stock,reorder_level,replenishment_date,inventory_updated_at,is_synthetic,available_stock
0,SYN-PROD-001,500,399,25,NaT,2011-12-10,True,101
1,SYN-PROD-002,107,6,65,NaT,2011-12-10,True,101
2,SYN-PROD-003,339,18,25,2011-12-18,2011-12-10,True,321
3,SYN-PROD-004,328,248,22,NaT,2011-12-10,True,80
4,SYN-PROD-005,114,82,34,NaT,2011-12-10,True,32
5,SYN-PROD-006,247,138,78,NaT,2011-12-10,True,109
6,SYN-PROD-007,89,7,42,NaT,2011-12-10,True,82
7,SYN-PROD-008,195,54,28,2011-12-14,2011-12-10,True,141
8,SYN-PROD-009,481,408,46,NaT,2011-12-10,True,73
9,SYN-PROD-010,46,6,34,NaT,2011-12-10,True,40


## 7. Generate reward catalogue

In [7]:
reward_ids = [f"SYN-REWARD-{i:03d}" for i in range(1, 21)]
reward_types = ["product"] * 15 + ["voucher"] * 5
reward_values = rng.integers(100, 2001, size=20)
points_required = rng.integers(500, 10001, size=20)

eligible_tier_options = [
    "Bronze,Silver",
    "Silver,Gold",
    "Gold,Platinum",
    "Bronze,Silver,Gold,Platinum"
]

reward_catalogue = pd.DataFrame({
    "reward_id": reward_ids,
    "reward_name": [
        "Everyday Essentials", "Home Comfort Kit", "Kitchen Starter Set",
        "Fitness Accessory", "Travel Utility Pack", "Desk Organiser",
        "Premium Bottle", "Wireless Accessory", "Personal Care Pack",
        "Coffee Time Set", "Storage Box Set", "Reading Light",
        "Eco Shopping Kit", "Lunch Box Set", "Home Decor Item",
        "Shopping Voucher 100", "Shopping Voucher 250",
        "Shopping Voucher 500", "Shopping Voucher 750", "Shopping Voucher 1000"
    ],
    "reward_type": reward_types,
    "reward_value": reward_values,
    "points_required": points_required,
    "eligible_tiers": [rng.choice(eligible_tier_options) for _ in reward_ids],
    "valid_from": pd.Timestamp("2011-11-01"),
    "valid_until": pd.Timestamp("2012-03-31"),
    "minimum_purchase_value": rng.choice([0, 250, 500, 1000], size=20),
    "status": ["active"] * 18 + ["inactive", "scheduled"],
    "campaign_id": [campaign_ids[i % len(campaign_ids)] for i in range(20)],
    "is_synthetic": True
})

display(reward_catalogue.head(10))

,reward_id,reward_name,reward_type,reward_value,points_required,eligible_tiers,valid_from,valid_until,minimum_purchase_value,status,campaign_id,is_synthetic
0,SYN-REWARD-001,Everyday Essentials,product,1730,1718,"Silver,Gold",2011-11-01,2012-03-31,0,active,SYN-CAMP-001,True
1,SYN-REWARD-002,Home Comfort Kit,product,911,9795,"Bronze,Silver",2011-11-01,2012-03-31,1000,active,SYN-CAMP-002,True
2,SYN-REWARD-003,Kitchen Starter Set,product,468,8629,"Gold,Platinum",2011-11-01,2012-03-31,1000,active,SYN-CAMP-003,True
3,SYN-REWARD-004,Fitness Accessory,product,1797,5819,"Bronze,Silver",2011-11-01,2012-03-31,500,active,SYN-CAMP-004,True
4,SYN-REWARD-005,Travel Utility Pack,product,1331,7858,"Bronze,Silver",2011-11-01,2012-03-31,250,active,SYN-CAMP-005,True
5,SYN-REWARD-006,Desk Organiser,product,1249,5146,"Bronze,Silver,Gold,Platinum",2011-11-01,2012-03-31,500,active,SYN-CAMP-001,True
6,SYN-REWARD-007,Premium Bottle,product,243,6213,"Bronze,Silver,Gold,Platinum",2011-11-01,2012-03-31,250,active,SYN-CAMP-002,True
7,SYN-REWARD-008,Wireless Accessory,product,953,9220,"Silver,Gold",2011-11-01,2012-03-31,500,active,SYN-CAMP-003,True
8,SYN-REWARD-009,Personal Care Pack,product,973,1246,"Gold,Platinum",2011-11-01,2012-03-31,1000,active,SYN-CAMP-004,True
9,SYN-REWARD-010,Coffee Time Set,product,598,3841,"Gold,Platinum",2011-11-01,2012-03-31,250,active,SYN-CAMP-005,True


## 8. Generate reward-product mapping

In [8]:
# Map product-type rewards to synthetic inventory products.
product_reward_catalogue = reward_catalogue[
    reward_catalogue["reward_type"] == "product"
].copy()

selected_products = rng.choice(
    product_ids,
    size=len(product_reward_catalogue),
    replace=False
)

reward_product_mapping = pd.DataFrame({
    "reward_id": product_reward_catalogue["reward_id"].to_numpy(),
    "product_id": selected_products,
    "minimum_stock_required": rng.integers(1, 11, size=len(product_reward_catalogue)),
    "inventory_constraint_type": rng.choice(
        ["available_stock", "replenishment_aware"],
        size=len(product_reward_catalogue),
        p=[0.70, 0.30]
    ),
    "is_synthetic": True
})

display(reward_product_mapping.head(10))

,reward_id,product_id,minimum_stock_required,inventory_constraint_type,is_synthetic
0,SYN-REWARD-001,SYN-PROD-008,2,available_stock,True
1,SYN-REWARD-002,SYN-PROD-025,6,available_stock,True
2,SYN-REWARD-003,SYN-PROD-028,7,replenishment_aware,True
3,SYN-REWARD-004,SYN-PROD-029,10,replenishment_aware,True
4,SYN-REWARD-005,SYN-PROD-026,5,available_stock,True
5,SYN-REWARD-006,SYN-PROD-018,3,available_stock,True
6,SYN-REWARD-007,SYN-PROD-012,6,available_stock,True
7,SYN-REWARD-008,SYN-PROD-006,4,available_stock,True
8,SYN-REWARD-009,SYN-PROD-023,2,replenishment_aware,True
9,SYN-REWARD-010,SYN-PROD-007,1,available_stock,True


## 9. Create an empty reward ledger template


In [9]:
ledger_columns = [
    "redemption_id",
    "idempotency_key",
    "request_hash",
    "customer_id",
    "reward_id",
    "campaign_id",
    "points_debited",
    "reward_cost",
    "redemption_status",
    "created_at",
    "is_synthetic"
]

reward_ledger = pd.DataFrame(columns=ledger_columns)

display(reward_ledger)
print("Ledger rows:", len(reward_ledger))
print("Ledger columns:", reward_ledger.columns.tolist())

,redemption_id,idempotency_key,request_hash,customer_id,reward_id,campaign_id,points_debited,reward_cost,redemption_status,created_at,is_synthetic


Ledger rows: 0
Ledger columns: ['redemption_id', 'idempotency_key', 'request_hash', 'customer_id', 'reward_id', 'campaign_id', 'points_debited', 'reward_cost', 'redemption_status', 'created_at', 'is_synthetic']


## 10. Validate relationships and business-data consistency

In [10]:
# Customer IDs must be drawn from the RFM dataset.
assert set(customer_loyalty_profiles["customer_id"]) == set(rfm["customer_id"])

# Reward mappings must point to existing rewards and inventory products.
assert set(reward_product_mapping["reward_id"]).issubset(set(reward_catalogue["reward_id"]))
assert set(reward_product_mapping["product_id"]).issubset(set(inventory_snapshot["product_id"]))

# Campaign references in rewards must exist.
assert set(reward_catalogue["campaign_id"]).issubset(set(campaign_budgets["campaign_id"]))

# Inventory arithmetic must be consistent.
assert (
    inventory_snapshot["available_stock"]
    == inventory_snapshot["on_hand_stock"] - inventory_snapshot["reserved_stock"]
).all()

# Budget arithmetic must be consistent.
assert (
    campaign_budgets["available_budget_before_approval"]
    == campaign_budgets["budget_limit"] - campaign_budgets["committed_amount"]
).all()

# Synthetic marker must be true for every generated record.
for dataset in [
    customer_loyalty_profiles,
    campaign_budgets,
    inventory_snapshot,
    reward_catalogue,
    reward_product_mapping
]:
    assert dataset["is_synthetic"].all()

print("All relationship and consistency checks: PASSED")

All relationship and consistency checks: PASSED


## 11. Save all six datasets as CSV

In [11]:
datasets = {
    "customer_loyalty_profiles.csv": customer_loyalty_profiles,
    "reward_catalogue.csv": reward_catalogue,
    "inventory_snapshot.csv": inventory_snapshot,
    "reward_product_mapping.csv": reward_product_mapping,
    "campaign_budgets.csv": campaign_budgets,
    "reward_ledger.csv": reward_ledger
}

for filename, dataset in datasets.items():
    file_path = output_dir / filename
    dataset.to_csv(file_path, index=False)
    print(f"Saved {filename}: {len(dataset)} rows")

print("\nAll six CSV files saved to:", output_dir)

Saved customer_loyalty_profiles.csv: 5878 rows
Saved reward_catalogue.csv: 20 rows
Saved inventory_snapshot.csv: 30 rows
Saved reward_product_mapping.csv: 15 rows
Saved campaign_budgets.csv: 5 rows
Saved reward_ledger.csv: 0 rows

All six CSV files saved to: d:\loyality_reward_optimization\data\processed\synthetic


## 12. Reload and verify saved CSV files

In [12]:
verification_rows = []

for filename, original_dataset in datasets.items():
    loaded_dataset = pd.read_csv(output_dir / filename)
    verification_rows.append({
        "file": filename,
        "original_rows": len(original_dataset),
        "saved_rows": len(loaded_dataset),
        "columns_match": list(original_dataset.columns) == list(loaded_dataset.columns),
        "row_count_match": len(original_dataset) == len(loaded_dataset)
    })

verification_report = pd.DataFrame(verification_rows)
display(verification_report)

if not verification_report["columns_match"].all():
    raise ValueError("At least one saved CSV has a column mismatch.")

if not verification_report["row_count_match"].all():
    raise ValueError("At least one saved CSV has a row-count mismatch.")

print("CSV reload verification: PASSED")

,file,original_rows,saved_rows,columns_match,row_count_match
0,customer_loyalty_profiles.csv,5878,5878,True,True
1,reward_catalogue.csv,20,20,True,True
2,inventory_snapshot.csv,30,30,True,True
3,reward_product_mapping.csv,15,15,True,True
4,campaign_budgets.csv,5,5,True,True
5,reward_ledger.csv,0,0,True,True


CSV reload verification: PASSED


In [4]:
import pandas as pd

df = pd.read_csv("../data/processed/customer_rfm_segments.csv")

print(df.columns.tolist())
print(df.head())

['customer_id', 'recency', 'frequency', 'monetary', 'cluster']
   customer_id  recency  frequency  monetary  cluster
0      12346.0      326         12  77556.46        1
1      12347.0        3          8   4921.53        1
2      12348.0       76          5   2019.40        1
3      12349.0       19          4   4428.69        1
4      12350.0      311          1    334.40        0
